### ACDP Weighting Cause of No SMOTE

In [32]:
import pandas as pd

# 1. Targeting Path
file_path = 'data/diabetes_preprocessed_privacy.csv'

# 2. Reading Database
df_privacy = pd.read_csv(file_path)

# 3. Verifikasi Data
print("System: Data berhasil di-load.")
df_privacy.head(100)

System: Data berhasil di-load.


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,DiffWalk,Sex,Age,Education,Income,Age_Group,BMI_Group,Income_Group,Education_Group,GenHlth_Group
0,0,1,1,1,40.0,1,0,0,0,0,...,1,0,9,4,3,MiddleAge,Obese,Low,MidEdu,Poor
1,0,0,0,0,25.0,1,0,0,1,0,...,0,0,7,6,1,Adult,Overweight,Low,HighEdu,Fair
2,0,1,1,1,28.0,0,0,0,0,1,...,1,0,9,4,8,MiddleAge,Overweight,High,MidEdu,Poor
3,0,1,0,1,27.0,0,0,0,1,1,...,0,0,11,3,6,Senior,Overweight,Middle,MidEdu,Good
4,0,1,1,1,24.0,0,0,0,1,1,...,0,0,11,5,4,Senior,Normal,Middle,HighEdu,Good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2,1,1,1,25.0,1,0,1,0,1,...,1,0,9,2,3,MiddleAge,Overweight,Low,LowEdu,Poor
96,2,0,0,1,32.0,0,0,0,1,0,...,0,0,3,5,3,Young,Obese,Low,HighEdu,Poor
97,0,1,0,1,44.0,0,0,0,1,1,...,0,0,9,4,6,MiddleAge,Obese,Middle,MidEdu,Fair
98,0,1,1,1,28.0,0,0,0,0,1,...,1,0,11,4,3,Senior,Overweight,Low,MidEdu,Good


### Logic

In [33]:
import pandas as pd
import numpy as np
from graphviz import Digraph
import os
os.environ["PATH"] += os.pathsep + 'C:/Program Files/Graphviz/bin'

# --- FIX 1: Fungsi Penentu Pemenang (Weighted Majority Vote) ---
def get_weighted_majority(target, weights):
    # Menghitung total bobot untuk setiap kelas (0, 1, 2)
    weighted_counts = weights.groupby(target).sum()
    return weighted_counts.idxmax() # Mengambil kelas dengan total bobot tertinggi

def build_acdp_tree_v2(data, target_col, features, weights, depth=0, max_depth=3):
    # Jika data kosong
    if len(data) == 0: return None
    
    # Base Case: Max depth atau semua data satu kelas
    if depth >= max_depth or len(data[target_col].unique()) == 1:
        return get_weighted_majority(data[target_col], weights.loc[data.index])

    # Scouting: Cari atribut terbaik (Weighted MI)
    best_ac = -1
    best_feat = None
    for feat in features:
        ac = weighted_mutual_info(data[feat], data[target_col], weights.loc[data.index])
        if ac > best_ac:
            best_ac = ac
            best_feat = feat
            
    if best_feat is None:
        return get_weighted_majority(data[target_col], weights.loc[data.index])

    node = {'feature': best_feat, 'children': {}, 'depth': depth}
    remaining_feats = [f for f in features if f != best_feat]
    
    for val in data[best_feat].unique():
        subset = data[data[best_feat] == val]
        node['children'][val] = build_acdp_tree_v2(
            subset, target_col, remaining_feats, weights, depth + 1, max_depth
        )
            
    return node

print("System: ACDP Tree Core Logic Compiled.")

System: ACDP Tree Core Logic Compiled.


### Visual

In [34]:
# --- FIX 2: Fungsi Real Visualisasi (Generate Nodes) ---
def visualize_tree(tree, feature_names):
    dot = Digraph(comment='ACDP Tree')
    dot.attr(rankdir='LR', size='10,10')
    
    def add_nodes(node, parent_name=None, edge_label=None):
        if not isinstance(node, dict): # Ini adalah Leaf
            color = 'green' if node == 0 else ('orange' if node == 1 else 'red')
            label = f"PREDICT: {int(node)}"
            node_name = str(np.random.rand())
            dot.node(node_name, label, shape='box', style='filled', fillcolor=color)
            if parent_name:
                dot.edge(parent_name, node_name, label=edge_label)
            return

        # Ini adalah Decision Node
        node_name = str(np.random.rand())
        dot.node(node_name, f"Split by:\n{node['feature']}", shape='ellipse')
        
        if parent_name:
            dot.edge(parent_name, node_name, label=edge_label)
            
        for val, child in node['children'].items():
            add_nodes(child, node_name, edge_label=str(val))

    add_nodes(tree)
    return dot

# --- Execution ---
# Pastikan df_privacy sudah ter-load
class_weights = {0.0: 1, 1.0: 60, 2.0: 10} # Buff kelas 1 lebih gila lagi
weights = df_privacy['Diabetes_012'].map(class_weights)

features_list = ['Age_Group', 'BMI_Group', 'Income_Group', 'Education_Group', 'GenHlth_Group']
my_acdp_tree_fix = build_acdp_tree_v2(df_privacy, 'Diabetes_012', features_list, weights, max_depth=3)

# Generate Real Nodes Visualization
graph = visualize_tree(my_acdp_tree_fix, features_list)

# 1. Buff Ketajaman (DPI)
graph.attr(dpi='300')

# 2. Perluasan Map (Size dalam inci, perbesar dari default 10,10)
graph.attr(size='15,20') 

# 3. Export to PDF (Sangat disarankan untuk pohon masif ini)
graph.render('acdp_tree_diagram_HD', format='pdf', view=True)

'acdp_tree_diagram_HD.pdf'